# Session 12: Introduction to LangGraph

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

So far, every LLM call in Project 03 has been one step: a report goes in, a triage comes out. Real applications then have to **decide** something: retry, fall back, ask a person, or act. This session is about expressing those decisions as a workflow, first in plain Python, then as a **LangGraph** state graph, and about knowing which one a problem deserves.

By the end of this notebook you will be able to:

1. **Explain** when a stateful graph is justified, and when an `if` statement in plain Python is enough.
2. **Define** typed graph state, nodes and edges, and **use** a reducer.
3. **Implement** conditional routing, including a loop with a limit.
4. **Run, stream and debug** a graph step by step, and **interpret** its errors.
5. **Test** a graph, and **assess** LangGraph against plain Python for the same workflow.

> **Kernel:** the course environment, after `uv sync`. LangGraph is in its `ai` group.
>
> **Cost: nothing.** No cell calls an LLM or the network: the workflow's dependencies are small fakes, so every run is deterministic.

## The engineering problem

The on-call team's monitoring sends **alerts**: a service, a severity, a message. Before anyone is woken up, each alert must be handled:

1. **Look up the owner** in the service catalogue. The catalogue is slow and sometimes times out, so retry once; if it still fails, don't guess.
2. **Decide:** page the owner for a critical alert, open a ticket for a warning, only log an info alert, and send anything without an owner to the triage desk.

That's a loop, a fallback and four branches. You've written logic like this many times in Java, C# or TypeScript. The question for this session is when such logic should stay ordinary code, and when it's worth turning into an explicit **graph**.

The cell below defines the alert and a fake catalogue that times out on its first `failures` lookups.

In [ ]:
import operator
import time
from dataclasses import dataclass
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


@dataclass(frozen=True)
class Alert:
    id: str
    service: str
    severity: Literal["critical", "warning", "info"]


class CatalogueTimeout(Exception):
    """The service catalogue didn't answer in time."""


class FlakyCatalogue:
    """A fake service catalogue: times out on its first `failures` lookups."""

    OWNERS = {"checkout-api": "platform", "orders-db": "database", "office-vpn": "network"}

    def __init__(self, failures: int = 0) -> None:
        self.failures = failures
        self.calls = 0

    def owner(self, service: str) -> str | None:
        self.calls += 1
        if self.calls <= self.failures:
            raise CatalogueTimeout(f"catalogue timed out looking up {service}")
        return self.OWNERS.get(service)  # None: the service isn't in the catalogue


CHECKOUT_DOWN = Alert("ALR-1", "checkout-api", "critical")
DB_WARNING = Alert("ALR-2", "orders-db", "warning")
MYSTERY = Alert("ALR-3", "label-printer", "critical")
print("ready")

## Part 1: Plain Python first

Here's the whole workflow as an ordinary function. Read it before running it: it's the baseline everything else is compared with.

In [ ]:
def handle_alert(alert: Alert, catalogue: FlakyCatalogue, *, max_attempts: int = 2) -> dict:
    steps = []
    owner = None
    for _ in range(max_attempts):
        steps.append("lookup")
        try:
            owner = catalogue.owner(alert.service)
            break
        except CatalogueTimeout:
            continue
    if owner is None:
        return {"action": "triage-desk", "owner": None, "steps": [*steps, "triage-desk"]}
    action = {"critical": "page", "warning": "ticket", "info": "log"}[alert.severity]
    return {"action": action, "owner": owner, "steps": [*steps, action]}


print(handle_alert(CHECKOUT_DOWN, FlakyCatalogue()))
print(handle_alert(DB_WARNING, FlakyCatalogue(failures=1)))
print(handle_alert(MYSTERY, FlakyCatalogue()))
print(handle_alert(CHECKOUT_DOWN, FlakyCatalogue(failures=5)))

Fourteen lines, readable top to bottom, trivially testable, no dependencies. **For this workflow as it stands, plain Python is the right answer.** Keep that in mind for the rest of the notebook.

What would make it harder? Requirements like these, which real workflows collect over time:

- *"Before paging for a critical alert at night, ask the incident commander to approve, and carry on when they answer, possibly hours later, possibly after a deployment restarted the process."*
- *"Show the on-call dashboard each step as it happens."*
- *"When it fails halfway, resume from the failed step instead of starting again."*
- *"Add enrichment, deduplication and a runbook lookup, some of them in parallel."*

Each is possible in plain Python, but you'd be writing a small workflow engine: saving state between steps, resuming, tracing. That's what LangGraph provides.

![Plain Python or a graph?](../../../assets/diagrams/linear-vs-graph-workflow.svg)

**How this works**

- The **pipeline** on the left has one path. Each step runs once, in order, so ordinary function calls already express it perfectly; a graph would add ceremony and a dependency, and nothing else.
- The **workflow** on the right has a loop (retry), a fallback, and several destinations. The interesting logic is no longer inside the steps but **between** them: which step runs next, and why.
- A graph makes those routes explicit. You can draw them, stream each step as it runs, put a limit on loops, and, with a checkpointer, pause for a person and resume later.
- The advice at the bottom is the decision rule for this session: **start with plain Python**, and move to a graph when the control flow becomes the hard part.

## Part 2: State, nodes and edges

LangGraph has three ideas. If you've used a state machine library (Spring Statemachine, Stateless in .NET, XState in JavaScript), they'll feel familiar, with one difference: the state is **data**, not just a label.

| Idea | What it is in LangGraph | In plain Python terms |
|---|---|---|
| **State** | A `TypedDict` describing the data every step can read | The local variables of `handle_alert` |
| **Node** | A function: it receives the state and returns an **update**, a dict of the keys it changes | One block of the function |
| **Edge** | Which node runs next: fixed, or chosen by a function | The order of the code, and its `if`s |

You **build** a graph with `StateGraph`, **compile** it, and then **invoke** it with an initial state. Here's the smallest useful graph: one node.

In [ ]:
class AlertState(TypedDict, total=False):
    alert: Alert
    owner: str | None


catalogue = FlakyCatalogue()


def lookup(state: AlertState) -> dict:
    return {"owner": catalogue.owner(state["alert"].service)}


builder = StateGraph(AlertState)
builder.add_node("lookup", lookup)
builder.add_edge(START, "lookup")
builder.add_edge("lookup", END)
graph = builder.compile()

final = graph.invoke({"alert": CHECKOUT_DOWN})
print(type(final).__name__, final)

**Explain:** what does `invoke` return? It's the **final state**, a plain `dict`: the input you passed, plus every update the nodes returned.

Three details matter:

- `total=False` makes every key optional, because the state fills up as nodes run. Before `lookup` runs, there's no `owner`.
- The node returned `{"owner": ...}` only. It didn't copy `alert` into its result: LangGraph **merges** each update into the state.
- `START` and `END` are special node names for "where the graph begins" and "stop here".

Notice also how the node got the catalogue: through the enclosing scope. Graph nodes receive only the state, so dependencies (a client, a triager, a catalogue) are usually passed in by building the graph inside a function, as Part 5 does. It's dependency injection by closure.

## Part 3: How updates are merged, and reducers

By default, an update **replaces** the key's value. That's usually right. But some keys should accumulate: a log of steps, a list of messages, a running total. For those, you attach a **reducer**: a function that combines the old value and the update.

**Predict:** the two nodes below each return `{"steps": [their name]}`. In the final state, what is `steps` for `ReplacingState`, and what is it for `AppendingState`?

In [ ]:
class ReplacingState(TypedDict, total=False):
    steps: list[str]


class AppendingState(TypedDict, total=False):
    steps: Annotated[list[str], operator.add]  # the reducer: old + new


def first(state: dict) -> dict:
    return {"steps": ["first"]}


def second(state: dict) -> dict:
    return {"steps": ["second"]}


for schema in (ReplacingState, AppendingState):
    builder = StateGraph(schema)
    builder.add_node("first", first)
    builder.add_node("second", second)
    builder.add_edge(START, "first")
    builder.add_edge("first", "second")
    builder.add_edge("second", END)
    print(f"{schema.__name__:<15}", builder.compile().invoke({"steps": []}))

With the reducer, `steps` is `old + new` after every update: `["first", "second"]`. Without it, the last writer wins. `Annotated[list[str], operator.add]` is ordinary Python typing; LangGraph reads the extra argument as the reducer. Any two-argument function works.

**Common Mistake: a misspelled key disappears silently.** An update with a key that isn't in the state schema is **ignored**, without an error:

In [ ]:
def typo(state: dict) -> dict:
    return {"step": ["oops"]}  # "step", not "steps"


builder = StateGraph(AppendingState)
builder.add_node("typo", typo)
builder.add_edge(START, "typo")
builder.add_edge("typo", END)
print(builder.compile().invoke({"steps": []}))

No `step`, no error, no warning. The bug shows up later, as a missing key in another node. Two defences: test what each node returns, and check the final state in tests rather than only the last value you care about.

**Common Mistake: mutating the state instead of returning an update.** A node receives the state object, so it *can* modify it:

In [ ]:
def sneaky(state: dict) -> dict:
    state["steps"].append("sneaky")  # mutates the list in place
    return {"steps": ["sneaky-update"]}


builder = StateGraph(AppendingState)
builder.add_node("sneaky", sneaky)
builder.add_edge(START, "sneaky")
builder.add_edge("sneaky", END)
print(builder.compile().invoke({"steps": []}))

The in-place change leaked into the result, alongside the proper update. It worked here, by accident; with checkpoints, retries or parallel branches, in-place changes are lost or applied twice. **Treat the state as read-only, and return everything you change.** It's the same discipline as reducers in Redux, if you've used it.

And a node must return a dict (or `None` for "no change"). Anything else fails fast:

In [ ]:
def returns_text(state: dict) -> str:
    return "done"


builder = StateGraph(AppendingState)
builder.add_node("returns_text", returns_text)
builder.add_edge(START, "returns_text")
builder.add_edge("returns_text", END)
try:
    builder.compile().invoke({"steps": []})
except InvalidUpdateError as error:
    print("InvalidUpdateError:", str(error).splitlines()[0])

## Part 4: Conditional edges

A fixed edge always goes to the same node. A **conditional edge** calls a **routing function** with the state, and goes to the node whose name it returns:

```python
builder.add_conditional_edges("lookup", after_lookup, ["lookup", "page", "ticket", "log", "triage-desk"])
```

The third argument declares the possible destinations. It looks optional, and technically it is. **Predict:** what happens to the graph's picture when you leave it out?

In [ ]:
class RouteState(TypedDict, total=False):
    severity: str
    steps: Annotated[list[str], operator.add]


def classify(state: RouteState) -> dict:
    return {"steps": ["classify"]}


def page(state: RouteState) -> dict:
    return {"steps": ["page"]}


def ticket(state: RouteState) -> dict:
    return {"steps": ["ticket"]}


def by_severity(state: RouteState) -> str:
    return "page" if state["severity"] == "critical" else "ticket"


def build(declare: bool):
    builder = StateGraph(RouteState)
    builder.add_node("classify", classify)
    builder.add_node("page", page)
    builder.add_node("ticket", ticket)
    builder.add_edge(START, "classify")
    if declare:
        builder.add_conditional_edges("classify", by_severity, ["page", "ticket"])
    else:
        builder.add_conditional_edges("classify", by_severity)
    builder.add_edge("page", END)
    builder.add_edge("ticket", END)
    return builder.compile()


for declare in (False, True):
    graph = build(declare)
    edges = sorted((edge.source, edge.target) for edge in graph.get_graph().edges)
    print(f"declared={declare!s:<5}", edges)
    print("   runs:", graph.invoke({"severity": "critical", "steps": []})["steps"])

Both graphs **run** correctly, but without declared destinations LangGraph can't know where `by_severity` might go. Its picture shows `classify → __end__`, and no edge to `page` or `ticket` at all. Anything that relies on the graph's structure (the drawing, your structure tests, a reviewer reading it) is then wrong.

Declare the destinations: as a list, as a dict mapping return values to node names (`{True: "page", False: "ticket"}`), or as a `Literal[...]` return type on the routing function, which LangGraph also reads.

**Common Mistake: a routing function returns a name that isn't a node.** It isn't caught when the graph is built, only when that route is taken. **Predict:** what happens with and without declared destinations?

In [ ]:
def misspelled(state: RouteState) -> str:
    return "paging"  # the node is called "page"


def build_misspelled(declare: bool):
    builder = StateGraph(RouteState)
    builder.add_node("classify", classify)
    builder.add_node("page", page)
    builder.add_node("ticket", ticket)
    builder.add_edge(START, "classify")
    if declare:
        builder.add_conditional_edges("classify", misspelled, ["page", "ticket"])
    else:
        builder.add_conditional_edges("classify", misspelled)
    builder.add_edge("page", END)
    builder.add_edge("ticket", END)
    return builder.compile()


print("undeclared:", build_misspelled(False).invoke({"severity": "critical", "steps": []}))
try:
    build_misspelled(True).invoke({"severity": "critical", "steps": []})
except KeyError as error:
    print("declared:   KeyError", error)

- **Undeclared:** no exception at all. LangGraph logs a warning (*"wrote to unknown channel branch:to:paging, ignoring it"*) and the run simply **ends** after `classify`, with nothing paged. In production that warning is easy to miss, and the alert is silently dropped.
- **Declared:** a `KeyError: 'paging'` from deep in LangGraph's code, at the moment the route is taken. Terse, but loud.

Another reason to declare destinations. When you see a `KeyError` whose value looks like a node name, check your routing functions first; a `Literal` return type with a type checker, or a test per route, catches it before it runs.

## Part 5: The alert workflow as a graph

Now the whole workflow from Part 1. Compare it with `handle_alert` as you read:

- `build_alert_graph` takes the catalogue and `max_attempts`, and its nodes reach them through the closure.
- `lookup` doesn't decide anything: it records what happened (`owner`, or the error) and counts attempts.
- The decisions live in two **routing functions**, which are ordinary functions of the state.
- `steps` records the path, through a reducer.

In [ ]:
class AlertState(TypedDict, total=False):
    alert: Alert
    owner: str | None
    error: str | None
    attempts: int
    action: str
    steps: Annotated[list[str], operator.add]


def build_alert_graph(catalogue: FlakyCatalogue, *, max_attempts: int = 2):
    def lookup(state: AlertState) -> dict:
        update = {"attempts": state.get("attempts", 0) + 1, "steps": ["lookup"]}
        try:
            return update | {"owner": catalogue.owner(state["alert"].service), "error": None}
        except CatalogueTimeout as error:
            return update | {"owner": None, "error": str(error)}

    def after_lookup(state: AlertState) -> Literal["lookup", "decide", "triage-desk"]:
        if state["error"] and state["attempts"] < max_attempts:
            return "lookup"  # try again
        if state["owner"] is None:
            return "triage-desk"  # timed out every time, or not in the catalogue
        return "decide"

    def decide(state: AlertState) -> dict:
        action = {"critical": "page", "warning": "ticket", "info": "log"}[state["alert"].severity]
        return {"action": action, "steps": ["decide"]}

    def act(name: str):
        def node(state: AlertState) -> dict:
            return {"action": name, "steps": [name]}

        return node

    builder = StateGraph(AlertState)
    builder.add_node("lookup", lookup)
    builder.add_node("decide", decide)
    for name in ("page", "ticket", "log", "triage-desk"):
        builder.add_node(name, act(name))
        builder.add_edge(name, END)
    builder.add_edge(START, "lookup")
    builder.add_conditional_edges("lookup", after_lookup)  # destinations from the Literal
    builder.add_conditional_edges(
        "decide", lambda state: state["action"], ["page", "ticket", "log"]
    )
    return builder.compile()


def run(alert: Alert, catalogue: FlakyCatalogue) -> dict:
    final = build_alert_graph(catalogue).invoke({"alert": alert, "attempts": 0, "steps": []})
    return {"action": final["action"], "owner": final["owner"], "steps": final["steps"]}


print(run(CHECKOUT_DOWN, FlakyCatalogue()))
print(run(DB_WARNING, FlakyCatalogue(failures=1)))
print(run(MYSTERY, FlakyCatalogue()))
print(run(CHECKOUT_DOWN, FlakyCatalogue(failures=5)))

**Run** both versions and compare the output with Part 1. The decisions are the same; the `steps` differ slightly, because the graph has an explicit `decide` step.

**Modify:** set `max_attempts=3` in `run`. Which alert's steps change, and how many times is the catalogue called? Then change `after_lookup` so that an alert whose service isn't in the catalogue goes to `decide` with the owner `"unknown"`. What else would you have to change?

**Explain:** the graph version is more than twice as long as `handle_alert`. What did you get for those extra lines? Look at the next two parts before answering.

The graph can draw itself. `draw_mermaid()` returns [Mermaid](https://mermaid.js.org/) text, which GitHub, many wikis and the Mermaid live editor render as a diagram. Dashed arrows (`-.->`) are conditional edges:

In [ ]:
print(build_alert_graph(FlakyCatalogue()).get_graph().draw_mermaid())

## Part 6: Running, streaming and debugging

`invoke` returns only the final state. When a run goes somewhere unexpected, you want to see **each step**. `stream` yields as the graph runs:

- `stream_mode="updates"`: one item per node, `{node_name: the update it returned}`. Best for "which node did what".
- `stream_mode="values"`: the whole state after each step. Best for "what did the state look like when it went wrong".

In [ ]:
graph = build_alert_graph(FlakyCatalogue(failures=1))
for update in graph.stream(
    {"alert": DB_WARNING, "attempts": 0, "steps": []}, stream_mode="updates"
):
    print(update)

**Engineering Insight:** this is the trace an on-call engineer wants when an alert was routed wrongly: each step, and exactly what it changed. In production you'd log these updates (without sensitive content) or send them to a tracing tool. LangSmith, from Session 10, records LangGraph runs if you enable it; the same data-protection questions apply.

### Errors inside a node

An exception raised in a node stops the graph and reaches your code **unchanged**: the same type, the same message, with the node's line in the traceback. You handle it as you would without LangGraph. Here, a catalogue with a bug of its own:

In [ ]:
class BrokenCatalogue(FlakyCatalogue):
    def owner(self, service: str) -> str | None:
        raise RuntimeError("catalogue client misconfigured")


try:
    build_alert_graph(BrokenCatalogue()).invoke({"alert": DB_WARNING, "attempts": 0, "steps": []})
except RuntimeError as error:
    print(type(error).__name__, "-", error)

That's why the `lookup` node catches only `CatalogueTimeout`, the failure the workflow knows how to handle, and lets everything else propagate. Catching `Exception` in a node and routing on it would turn a configuration bug into a quiet "triage-desk" for every alert.

### Loops need a limit

A routing bug can make a graph loop forever. **Predict:** this graph's routing function always sends `lookup` back to itself. What happens when it runs, and how many times is the catalogue called?

In [ ]:
def build_looping_graph(catalogue: FlakyCatalogue):
    def lookup(state: AlertState) -> dict:
        return {"owner": catalogue.owner(state["alert"].service), "steps": ["lookup"]}

    builder = StateGraph(AlertState)
    builder.add_node("lookup", lookup)
    builder.add_edge(START, "lookup")
    builder.add_conditional_edges("lookup", lambda state: "lookup", ["lookup"])  # the bug
    return builder.compile()


catalogue = FlakyCatalogue()
started = time.perf_counter()
try:
    build_looping_graph(catalogue).invoke({"alert": DB_WARNING, "steps": []})
except GraphRecursionError as error:
    print(str(error).splitlines()[0])
print(f"catalogue calls: {catalogue.calls:,} in {time.perf_counter() - started:.1f} s")

LangGraph stops the run with `GraphRecursionError` after a **recursion limit** of steps (LangGraph calls them *super-steps*). What's the default? Older tutorials say 25, and at the time of writing the official documentation says 1,000. The version this course uses stopped at **10,007**, which is what its source code sets (overridable with the `LANGGRAPH_DEFAULT_RECURSION_LIMIT` environment variable). Measure, don't trust: defaults change between versions.

Here that meant 10,007 catalogue calls, made in a second or two. **If each step were an LLM call, that's 10,007 paid requests** before anything stops it.

Set the limit yourself, per run, to just above the longest path your workflow can legitimately take:

In [ ]:
catalogue = FlakyCatalogue()
try:
    build_looping_graph(catalogue).invoke(
        {"alert": DB_WARNING, "steps": []}, {"recursion_limit": 5}
    )
except GraphRecursionError as error:
    print(str(error).splitlines()[0])
print("catalogue calls:", catalogue.calls)

How tight can it be? The alert workflow's longest path is `max_attempts` lookups and one action: 3 nodes with `max_attempts=2`. **Run** the next cell: it tries ever larger limits on that path.

In [ ]:
for limit in range(1, 6):
    try:
        build_alert_graph(FlakyCatalogue(failures=9)).invoke(
            {"alert": DB_WARNING, "attempts": 0, "steps": []}, {"recursion_limit": limit}
        )
        print(f"limit {limit}: finished")
        break
    except GraphRecursionError:
        print(f"limit {limit}: GraphRecursionError")

A path of 3 nodes needed a limit of **4**: one more than the number of nodes that run, as measured here. Don't rely on remembering that rule: compute the limit from your longest path, and **test** it, as the workbook does.

A loop counter in the state (`attempts`) is the first defence: it makes the intended limit part of the logic. The recursion limit is the safety net for the loops you didn't intend.

## Part 7: Pausing for a person (awareness)

One requirement from Part 1 was: *ask a person before paging, and carry on when they answer*. This is where a graph framework earns its keep. A node calls `interrupt(...)`; the run stops and **saves its state** with a **checkpointer**; later, you resume the same run (identified by a `thread_id`) with the person's answer.

In [ ]:
class ApprovalState(TypedDict, total=False):
    service: str  # plain values: the checkpointer saves the state
    decision: str


def ask_commander(state: ApprovalState) -> dict:
    answer = interrupt({"question": f"Page the owner of {state['service']}?"})
    return {"decision": "paged" if answer == "yes" else "held"}


builder = StateGraph(ApprovalState)
builder.add_node("ask_commander", ask_commander)
builder.add_edge(START, "ask_commander")
builder.add_edge("ask_commander", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "ALR-1"}}
first = graph.invoke({"service": "checkout-api"}, config)
print("paused:", first["__interrupt__"][0].value)
print("waiting at:", graph.get_state(config).next)

resumed = graph.invoke(Command(resume="yes"), config)
print("resumed:", resumed["decision"])

- The first `invoke` ran until `interrupt`, saved the state under the thread `ALR-1`, and returned with an `__interrupt__` entry holding the question.
- `get_state(config).next` shows where the run is waiting.
- `Command(resume="yes")` resumed it: `interrupt(...)` returned `"yes"` inside the node, which then finished. The node runs again from its start when it resumes, so the code before `interrupt` must be safe to repeat.

The checkpointer **serialises** the state, which is why it holds plain values here. Custom classes in checkpointed state trigger a warning in this version, and future versions will block them unless you allow them explicitly. `InMemorySaver` loses everything when the process exits. In production, checkpoints go to a database, through a separate checkpointer package. That's beyond this course: the point is to recognise that pausing, persisting and resuming is a strong reason to use a workflow framework, and a weak reason to write your own.

## Part 8: Testing a graph

A graph is code, and you test it like code. Four layers, cheapest first:

1. **Routing functions and node logic as plain functions.** They're functions of a dict: no graph needed. In the workbook, the business rules (`review_reasons`) are an ordinary function that both versions share.
2. **The graph through its public entry point**, with fake dependencies injected through the build function: `invoke`, then assert on the final state, including the path taken.
3. **The structure**: the node names and edges from `get_graph()`. It catches undeclared destinations and wiring mistakes that a test of one path wouldn't.
4. **A contract with the plain-Python version**, when you have one: the same tests, run against both, as in Session 10.

In [ ]:
def test_retry_then_page():
    final = build_alert_graph(FlakyCatalogue(failures=1)).invoke(
        {"alert": CHECKOUT_DOWN, "attempts": 0, "steps": []}
    )
    assert final["steps"] == ["lookup", "lookup", "decide", "page"]
    assert final["owner"] == "platform"


def test_every_route_is_declared():
    edges = {(e.source, e.target) for e in build_alert_graph(FlakyCatalogue()).get_graph().edges}
    assert ("lookup", "lookup") in edges
    assert {("decide", "page"), ("decide", "ticket"), ("decide", "log")} <= edges


def test_same_decision_as_plain_python():
    for alert, failures in [(CHECKOUT_DOWN, 0), (DB_WARNING, 1), (MYSTERY, 0), (CHECKOUT_DOWN, 5)]:
        plain = handle_alert(alert, FlakyCatalogue(failures))
        graph = run(alert, FlakyCatalogue(failures))
        assert (plain["action"], plain["owner"]) == (graph["action"], graph["owner"]), alert


for test in (
    test_retry_then_page,
    test_every_route_is_declared,
    test_same_decision_as_plain_python,
):
    test()
    print("passed:", test.__name__)

## Part 9: The workflow you'll build

The workbook adds this routing workflow to **Project 03**. After a report is triaged, the workflow decides whether to page the owning team, queue the report, or send it to a person, and it handles the LLM's failures: invalid output is retried, an unavailable provider falls back to the keyword rules, and a refusal goes straight to review. You build it twice, in plain Python and as a graph, and the same tests run against both.

![The routing workflow as a LangGraph graph](../../../assets/diagrams/langgraph-state-transitions.svg)

**How this works**

- Every run starts at **triage**, which calls the LLM triager and records how it went (`status`, `error`) instead of deciding what to do next. The decision belongs to the routing function after it.
- That routing function has four outcomes: **check** a good triage, **retry** invalid output while `attempts` is below the maximum, use the **fallback** when the provider is unavailable or retries are exhausted, and go straight to **review** on a refusal, because keyword rules shouldn't overrule a model that declined.
- **check** applies the business rules (an owning team, an affected system for a P1, and so on), and a second routing function picks **page**, **queue** or **review**. Every path ends at one of those three.
- The state panel shows who writes what. Every key is replaced by updates except `path`, whose reducer appends, so each run records its route, such as `["triage", "triage", "check", "page"]`.
- The longest path is every attempt, then fallback, check and an action. The recursion limit is derived from it, so a routing bug can't loop through paid LLM calls.

## Part 10: Plain Python or LangGraph?

| | Plain Python | LangGraph |
|---|---|---|
| **Reading the logic** | Top to bottom, in one function | Nodes, plus wiring; the routes are explicit and drawable |
| **Dependencies** | None | `langgraph` and its packages (LangChain core, checkpoint libraries) |
| **Debugging** | Your debugger and stack traces | The same, plus `stream` per step; tracebacks go through LangGraph's code |
| **Testing** | Call the function | Invoke the graph; also test routing functions and structure |
| **Loops** | `for` with a counter | Edges back to a node, plus a recursion limit you must set |
| **Pause and resume, persistence** | You build it | `interrupt` and a checkpointer |
| **Streaming each step, tracing** | You build it | `stream`, and LangSmith if you enable it |
| **Parallel branches** | `asyncio.gather` or threads (Session 08) | Fan-out edges, with reducers merging results |

If you've used workflow engines such as AWS Step Functions or Temporal, LangGraph sits much lower: a library inside your process, not a service. Its closest relatives are state-machine libraries, with the state being application data and the steps often being LLM calls.

**A decision rule:** if the workflow fits in one function you can read in a minute, has no need to pause, persist or resume, and nobody needs to watch it step by step, keep plain Python. Reach for a graph when you need those things, or when the routing itself has become the part that's hard to change safely. And whichever you choose, keep the business rules in plain functions that both could use.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| A graph for a straight pipeline | More code and a dependency, nothing gained | Plain functions until the control flow is the hard part |
| Undeclared conditional destinations | The drawing and structure checks are wrong | A list, a mapping, or a `Literal` return type |
| A misspelled key in an update | Silently ignored | Test node outputs and the final state |
| Mutating the state in a node | Changes leak, or are lost with checkpoints | Return an update; treat the state as read-only |
| A list key without a reducer | Each update replaces the list | `Annotated[list[...], operator.add]` |
| No recursion limit on a graph with loops | Up to 10,007 steps, possibly paid calls | A loop counter in the state, and a tight `recursion_limit` |
| Catching every exception in a node | Bugs become ordinary routes | Catch what the workflow handles; let the rest propagate |
| Decisions inside nodes and routing functions mixed | Hard to test, hard to read | Nodes record what happened; routing functions decide |
| Business rules only inside the graph | Can't reuse or test them alone | Plain functions, called by the nodes |

## Practical engineering application

AI applications are full of small workflows around model calls: validate the output, retry with feedback, fall back to a cheaper model or rules, escalate to a person, and record what happened. Most should be plain Python. LangGraph is worth its cost when the workflow needs to **pause for people**, **survive restarts**, **show its progress**, or when its routing has grown complex enough that an explicit, drawable, testable graph is clearer than nested `if`s. Agents, which Session 12 deliberately doesn't cover, are built on the same pieces: a node that calls a model, a node that runs tools, and a routing function in a loop.

## Key takeaways

- Start with plain Python. Use a graph when the control flow, pausing, persistence or step-by-step visibility becomes the hard part.
- State is a `TypedDict`; nodes return **updates** with only the keys they change; reducers decide how updates combine.
- Conditional edges call a routing function. Declare its destinations.
- Nodes record what happened; routing functions decide what's next; business rules live in plain functions.
- Exceptions propagate unchanged. Catch only what the workflow handles.
- Every loop needs a counter in the state and a recursion limit computed from the longest legitimate path. The default is 10,007.
- Test the routing functions, the graph through `invoke`, its structure, and, where you have one, the plain-Python equivalent.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then build the routing workflow in the workbook.